# 02 特徴量の分布確認

`python -m ct125 process --all` の出力 `data/processed/<ride_id>/windows.parquet` を読み、特徴量の分布と速度との関係を目視する。
処理の本体は `src/ct125/`（io / clean / gps / features / cli）。緯度経度の実値は表示しない。

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
plt.rcParams["font.family"] = ["Meiryo", "Yu Gothic", "MS Gothic", "sans-serif"]; plt.rcParams["axes.unicode_minus"] = False
PROC = Path("../data/processed")
W = pd.concat([pd.read_parquet(p) for p in sorted(PROC.glob("ride_*/windows.parquet"))], ignore_index=True)
W = W.drop(columns=["lat", "lon"])  # このノートブックでは座標を使わない
W.groupby("ride_id").agg(windows=("win", "size"), has_gps=("has_gps", "mean"), t_abs=("t_abs", lambda s: s.notna().mean()),
                         edge=("is_edge", "sum"), clipped=("is_clipped", "sum"), rms_med=("rms", "median"), rms_p99=("rms", lambda s: s.quantile(.99)),
                         n_samples_min=("n_samples", "min"), n_samples_max=("n_samples", "max")).round(3)

- 往復の走行（ride_004/005）は `has_gps` が 0.9 前後、GPS なし走行（ride_006〜009）は全窓 `has_gps=False`・`t_abs` なし（docs/analysis_policy.md 3.）。
- `n_samples` は 397〜400 でそろっている。

## 1. 特徴量の分布（走行ごと、is_edge を除く）

In [ ]:
M = W[~W.is_edge]
cols = ["log_rms", "p2p", "crest", "jerk_rms", "band_1_5", "band_5_15", "band_15_40"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7)); axes = axes.ravel()
for ax, c in zip(axes, cols):
    for rid, g in M.groupby("ride_id"):
        if rid in ("ride_004", "ride_005", "ride_007", "ride_009"):
            ax.hist(g[c].dropna(), bins=50, histtype="step", density=True, label=rid)
    ax.set_title(c)
axes[0].legend(); axes[-1].axis("off"); plt.tight_layout()

## 2. 速度と振動（ride_004 / ride_005、has_gps かつ is_edge でない窓）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, rid in zip(axes, ["ride_004", "ride_005"]):
    g = M[(M.ride_id == rid) & M.has_gps]
    ax.scatter(g.speed_kmh, g.log_rms, s=6, alpha=0.5)
    rho_all, _ = spearmanr(g.speed_kmh, g.log_rms)
    mv = g[g.speed_kmh >= 5]; rho_mv, _ = spearmanr(mv.speed_kmh, mv.log_rms)
    med = g.groupby(pd.cut(g.speed_kmh, np.arange(0, 105, 5)), observed=True).log_rms.median()
    ax.plot([iv.mid for iv in med.index], med.values, "r-o", ms=4, label="5 km/h ビンの中央値")
    ax.set_title(f"{rid}: Spearman 全窓 {rho_all:.2f} / 走行中（≥5 km/h）{rho_mv:.2f}  n={len(g)}")
    ax.set_xlabel("speed_kmh"); ax.set_ylabel("log_rms"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()

- 速度が上がるほど `log_rms` が上がる正の傾向がある（完了条件）。停止（<5 km/h）と走行の差が大きく、走行中だけに絞っても正。
- ride_004 の 60 km/h 超の窓（右端）は傾向から外れて低い。これは 14:15:30〜48 の「98 km/h」区間で、振動から見ると実速度ではない可能性が高い。

## 3. 時系列（ride_004：RMS と速度、帯域比）

In [ ]:
g = W[W.ride_id == "ride_004"]
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
axes[0].plot(g.t_abs, g.rms, lw=.8, label="rms [g]"); axes[0].fill_between(g.t_abs, 0, g.rms.max(), where=g.is_edge, color="gray", alpha=.15, label="is_edge")
axes[0].legend(loc="upper left"); axes[0].set_ylabel("g")
axes[1].plot(g.t_abs, g.speed_kmh, lw=.8, color="tab:green", label="speed_kmh"); axes[1].fill_between(g.t_abs, 0, 100, where=~g.has_gps, color="red", alpha=.12, label="has_gps=False")
axes[1].legend(loc="upper left"); axes[1].set_ylabel("km/h")
axes[2].stackplot(g.t_abs, g.band_1_5, g.band_5_15, g.band_15_40, labels=["1-5 Hz", "5-15 Hz", "15-40 Hz"], alpha=.8)
axes[2].legend(loc="upper left"); axes[2].set_ylabel("帯域比"); axes[2].set_xlabel("JST")
plt.tight_layout()

## 4. GPS なし走行（ride_007）：相対時間だけのタイムライン

In [ ]:
g = W[W.ride_id == "ride_007"]
fig, ax = plt.subplots(figsize=(15, 3.5))
ax.plot(g.t_rel / 60, g.rms, lw=.7); ax.scatter(g.t_rel[g.is_clipped] / 60, g.rms[g.is_clipped], color="red", s=30, label="is_clipped（15.9 g 以上を含む窓）")
ax.set_xlabel("走行開始からの分（t_abs なし）"); ax.set_ylabel("rms [g]"); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f"ride_007: 窓 {len(g)}, has_gps={g.has_gps.mean():.0%}")
plt.tight_layout()

## まとめ

- 9 本すべてで `process` が通り、`windows.parquet` にdocs/analysis_policy.md 2.3 の列＋帯域比＋`is_clipped` が揃う。
- 往復の走行は `has_gps` 0.92 / 0.87（無効化の内訳は `data/interim/<ride>/gps.parquet` の `invalid_reason`）。GPS なし走行は全窓 `has_gps=False`。
- 速度と `log_rms` は正の相関（ride_004: 0.74、ride_005: 0.65。走行中のみで 0.39 / 0.37）。